In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
base_dir = os.getcwd()
clean_data_dir = os.path.join(base_dir,'Ex2','Bangalore_House_Price_data','clean_data.csv')


In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [4]:
df = pd.read_csv(clean_data_dir)

In [5]:
df.shape

(8148, 6)

In [6]:
df.head()

,bath,balcony,price,total_sqft_float,bhk,price_per_sqft
0,2.0,1.0,39.07,1056.0,2,3699.810606
1,5.0,3.0,120.00,2600.0,4,4615.384615
2,2.0,3.0,62.00,1440.0,3,4305.555556
3,2.0,1.0,51.00,1200.0,2,4250.000000
4,2.0,1.0,38.00,1170.0,2,3247.863248


In [7]:
X = df.drop(columns = ['price'])
y = df['price']

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 36)

In [9]:
X_train.describe()

,bath,balcony,total_sqft_float,bhk,price_per_sqft
count,6518.000000,6518.000000,6518.000000,6518.000000,6518.000000
mean,2.435716,1.626879,1530.694984,2.517950,5796.869845
std,0.880331,0.777204,968.892544,0.836431,2566.763822
min,1.000000,0.000000,350.000000,1.000000,2131.115460
25%,2.000000,1.000000,1123.000000,2.000000,4306.265682
50%,2.000000,2.000000,1300.000000,2.000000,5250.086386
75%,3.000000,2.000000,1680.000000,3.000000,6472.338134
max,9.000000,3.000000,30400.000000,9.000000,35000.000000


có thể thấy giá trị các feature rất chênh lệch vì chúng có các thang đo khác nhau. đối với 1 số mô hình thì việc này làm mô hình tệ đi, vì thế cần chuẩn hóa các feature về cùng 1 thang đo
* StandardScaler biến tập giá trị thành tập giá trị có mean là 0 và độ lệch chuẩn là 1, sử dụng công thức z = (x- mean)/std

In [10]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
sc.fit(X_train)
X_train = sc.transform(X_train)
X_test = sc.transform(X_test)
# sử dụng sc fit cho duy nhất Xtrain và dùng chung cho cả X_test là vì 
# nếu fit cho test riêng thì có thể tạo ra các bộ tham số mean và std khác
# sử dụng 2 bộ tham số khác nhau, qua 2 hàm biến đổi khác nhau làm sai ý nghĩa số học

In [11]:
X_train # đầu ra của transform là array 2d

array([[ 1.77706232, -0.80664446,  1.17184524,  0.57636177,  0.66431372],
       [-0.49498401, -0.80664446, -0.5477756 , -0.61928623, -1.13336996],
       [-0.49498401,  0.48011731, -0.52506753, -0.61928623, -0.73365534],
       ...,
       [-0.49498401, -0.80664446, -0.26702118, -0.61928623,  0.65286216],
       [-0.49498401, -0.80664446, -0.32069482, -0.61928623, -0.45419517],
       [-0.49498401,  0.48011731, -0.1245796 ,  0.57636177, -0.2413983 ]],
      shape=(6518, 5))

In [12]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error
lr = LinearRegression()
lr_lasso = Lasso()
lr_ridge = Ridge()

In [13]:
def rmse(y_test, y_pred):
    return np.sqrt(mean_squared_error(y_test, y_pred))

In [14]:
lr.fit(X_train, y_train)
lr_score = lr.score(X_test, y_test) 
lr_rmse = rmse(y_test, lr.predict(X_test))
lr_score, lr_rmse
#score trả về accuracy trong độ chính xác và r2 cho hồi quy

(0.8725904294004125, np.float64(31.270039963686422))

In [17]:
lr_rmse/y_test.mean()

np.float64(0.33513220660244386)

In [ ]:
# Lasso
lr_lasso.fit(X_train, y_train)
lr_lasso_score=lr_lasso.score(X_test, y_test) # balcony trong bài được fillna bởi mode
lr_lasso_rmse = rmse(y_test, lr_lasso.predict(X_test))
lr_lasso_score, lr_lasso_rmse
# balcony trong tài liệu được fill bởi mean trong bài được fill bởi mean trong khi 
# ban công nên là số nguyên điều này tạo ra các giá trị vô lí gây giảm độ chính xác

(0.8734059449403203, np.float64(31.169803611077715))

In [21]:
lr_lasso_rmse/y_test.mean()

np.float64(0.3340579377473178)

In [22]:
 # Ridge
lr_ridge.fit(X_train, y_train)
lr_ridge_score = lr_ridge.score(X_test, y_test) 
lr_ridge_rmse = rmse(y_test, lr_ridge.predict(X_test))
lr_ridge_score, lr_ridge_rmse

(0.8726102898526986, np.float64(31.26760270038949))

In [23]:
lr_ridge_rmse/y_test.mean()

np.float64(0.33510608557964633)

In [36]:
from sklearn.ensemble import RandomForestRegressor
tree_no = [i for i in range (100,1001,100)]

for i in tree_no:
    rfr = RandomForestRegressor(n_estimators = i, max_features = (int)(np.ceil(X_train.shape[1]/3)))
    rfr.fit(X_train,y_train)
    rfr_score=rfr.score(X_test,y_test)
    rfr_rmse = rmse(y_test, rfr.predict(X_test))
    print(f"Number of estimator: {i}, rfr_score: {rfr_score}, rfr_rmse: {rfr_rmse}")


Number of estimator: 100, rfr_score: 0.9817188501294538, rfr_rmse: 11.844839076371816
Number of estimator: 200, rfr_score: 0.9795565483437793, rfr_rmse: 12.525772572551693
Number of estimator: 300, rfr_score: 0.9806401260279465, rfr_rmse: 12.189297396868563
Number of estimator: 400, rfr_score: 0.981273410303283, rfr_rmse: 11.988276686997287
Number of estimator: 500, rfr_score: 0.977367390767043, rfr_rmse: 13.17937219091734
Number of estimator: 600, rfr_score: 0.981061379434481, rfr_rmse: 12.055953987537208
Number of estimator: 700, rfr_score: 0.9809161905950474, rfr_rmse: 12.10207793867336
Number of estimator: 800, rfr_score: 0.9806974651125451, rfr_rmse: 12.17123319272793
Number of estimator: 900, rfr_score: 0.9809273944409561, rfr_rmse: 12.098524934131623
Number of estimator: 1000, rfr_score: 0.9815646939221632, rfr_rmse: 11.894675171833319


In [59]:
import joblib
os.makedirs(os.path.join(base_dir,'Ex2','Bangalore_House_Price_data','model'), exist_ok = True)
model_dir = os.path.join(base_dir,'Ex2','Bangalore_House_Price_data','model','random_forest.pkl')
joblib.dump(rfr, model_dir)

['c:\\Weekly_Hw_GR1\\Ex2\\Bangalore_House_Price_data\\model\\random_forest.pkl']

In [60]:
rfr_load = joblib.load(model_dir)
rfr_load.predict(X_test)

array([ 35.15683,  57.75077,  46.76696, ..., 195.256  , 253.385  ,
        90.29914], shape=(1630,))